<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_10_3_transformer_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 10: Zeitreihen in PyTorch**

* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 10 Material

* Teil 10.1: Zeitreihen-Datenkodierung für Deep Learning, PyTorch [[Video]](https://www.youtube.com/watch?v=CZi5Avp6p1s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=CZi5Avp6p1s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.2: LSTM-basierte Zeitreihen mit PyTorch [[Video]](https://www.youtube.com/watch?v=hIQLy5zCgH4&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=hIQLy5zCgH4&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* **Teil 10.3: Transformer-basierte Zeitreihen mit PyTorch** [[Video]](https://www.youtube.com/watch?v=NGzQpphf_Vc&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=NGzQpphf_Vc&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.4: Saisonalität und Trend [[Video]](https://www.youtube.com/watch?v=HOkxoLaUF9s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=HOkxoLaUF9s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.5: Vorhersagen mit Meta Prophet [[Video]](https://www.youtube.com/watch?v=MzjMVsz0GyA&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=MzjMVsz0GyA&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)

# Google CoLab-Anweisungen

Der folgende Code überprüft, ob Google CoLab installiert ist, und richtet die richtigen Hardwareeinstellungen für PyTorch ein.


In [1]:
try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

Note: using Google CoLab
Using device: cuda


# Teil 10.5: Transformatoren für Zeitreihen in PyTorch

Die transformative Landschaft des Deep Learning hat in der jüngsten Vergangenheit enorme Fortschritte gemacht, insbesondere im Bereich der Verarbeitung natürlicher Sprache (NLP). Im Mittelpunkt dieser Revolution stand die Einführung von Transformer-Architekturen, die mit ihren Aufmerksamkeitsmechanismen die Grenzen des Machbaren bei Aufgaben wie maschineller Übersetzung, Stimmungsanalyse und Sprachmodellierung verschoben haben. Obwohl Transformer zunächst hauptsächlich im Bereich NLP an Bedeutung gewannen, ist ihre Anwendbarkeit nicht nur auf Textdaten beschränkt. Es besteht ein wachsendes Interesse daran, diese Modelle für Zeitreihenvorhersagen zu nutzen – eine Herausforderung, die zwar zahlenmäßig anders ist, aber konzeptionell dem Verstehen von Sequenzen in Sprache ähnelt.

Bei der Zeitreihenvorhersage geht es oft darum, zukünftige Werte auf der Grundlage historischer Daten vorherzusagen. Dies kann die Vorhersage von Aktienkursen, Wettermustern oder sogar des Stromverbrauchs in einer Region umfassen. Im Kern handelt es sich dabei um eine Sequenz-zu-Sequenz-Aufgabe, bei der die vergangenen Werte eine Eingabesequenz und die vorherzusagenden zukünftigen Werte eine Ausgabesequenz bilden. Betrachten wir nun die Ähnlichkeiten mit der maschinellen Übersetzung in NLP, wo eine Eingabesequenz (Satz) in einer Sprache in eine Ausgabesequenz in einer anderen Sprache übersetzt wird. In beiden Szenarien muss das Modell Muster, Abhängigkeiten und Kontext über Sequenzen hinweg erkennen.

In diesem Kapitel werden die Nuancen der Verwendung von PyTorch-Transformatoren zur Vorhersage von Zeitreihen eingehend behandelt. Wir beginnen diese Reise, indem wir zunächst ein grundlegendes Verständnis dafür entwickeln, wie Transformatoren im NLP-Bereich funktionieren, bevor wir uns ihrer Anpassung für numerische Sequenzen widmen. Durch die Gegenüberstellung dieser beiden Anwendungen erhalten die Leser ein umfassendes Verständnis für die Vielseitigkeit des Transformators und die subtilen Überlegungen, die beim Übergang von Text zu Zeit erforderlich sind.



## Laden von Sonnenfleckendaten für eine Transformator-Zeitreihe

Wir werden uns dieselben Sonnenfleckendaten wie im vorherigen Abschnitt ansehen. Dieses Mal werden wir jedoch einen Transformator zur Vorhersage verwenden. Die für dieses Beispiel benötigten Datendateien finden Sie am folgenden Speicherort.

* [Sunspot Data Files](http://www.sidc.be/silso/datafiles#total)
* [Download Daily Sunspots](http://www.sidc.be/silso/INFO/sndtotcsv.php) – 1.1.1818 bis heute.

Wir verwenden den folgenden Code, um die Sonnenfleckendatei zu laden:


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

names = [
    "year",
    "month",
    "day",
    "dec_year",
    "sn_value",
    "sn_error",
    "obs_num",
    "unused1",
]
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/SN_d_tot_V2.0.csv",
    sep=";",
    header=None,
    names=names,
    na_values=["-1"],
    index_col=False,
)

Die Vorverarbeitung der Daten erfolgt wie im vorherigen Abschnitt beschrieben. Wir verwenden Daten vor dem Jahr 2000 als Training, der Rest dient zur Validierung.

In [3]:
# Datenvorverarbeitung
start_id = max(df[df["obs_num"] == 0].index.tolist()) + 1
df = df[start_id:].copy()
df["sn_value"] = df["sn_value"].astype(float)
df_train = df[df["year"] < 2000]
df_test = df[df["year"] >= 2000]

spots_train = df_train["sn_value"].to_numpy().reshape(-1, 1)
spots_test = df_test["sn_value"].to_numpy().reshape(-1, 1)

scaler = StandardScaler()
spots_train = scaler.fit_transform(spots_train).flatten().tolist()
spots_test = scaler.transform(spots_test).flatten().tolist()

Genau wie wir es im vorherigen Abschnitt für LSTM getan haben, zerlegen wir die Daten erneut in Sequenzen.

In [4]:
# Sequenzdatenaufbereitung
SEQUENCE_SIZE = 10


def to_sequences(seq_size, obs):
    x = []
    y = []
    for i in range(len(obs) - seq_size):
        window = obs[i : (i + seq_size)]
        after_window = obs[i + seq_size]
        x.append(window)
        y.append(after_window)
    return torch.tensor(x, dtype=torch.float32).view(-1, seq_size, 1), torch.tensor(
        y, dtype=torch.float32
    ).view(-1, 1)


x_train, y_train = to_sequences(SEQUENCE_SIZE, spots_train)
x_test, y_test = to_sequences(SEQUENCE_SIZE, spots_test)

# Einrichten von Datenladern für die Stapelverarbeitung
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Positionskodierung für Transformatoren

Im Bereich der Transformer-Architektur ist die Fähigkeit, die Reihenfolge der Sequenz zu berücksichtigen, eine entscheidende Komponente, die den Erfolg des Modells sicherstellt. Im Gegensatz zu herkömmlichen RNNs oder LSTMs, die Sequenzen schrittweise verarbeiten und ihre Reihenfolge von Natur aus einhalten, verarbeiten Transformer alle Token in einer Sequenz gleichzeitig. Diese parallele Verarbeitung steigert zwar die Rechenleistung erheblich und ermöglicht eine effektivere Erfassung von Abhängigkeiten über große Entfernungen, bedeutet aber auch, dass Transformer in ihrer nativen Form die Position oder Reihenfolge der Token in einer Sequenz nicht berücksichtigen. Hier kommt das Konzept der PositionsKodierung ins Spiel.

Positionskodierung ist ein Mechanismus, der dem Transformator Informationen über die Position von Token innerhalb einer Sequenz liefert. Im Wesentlichen fügt sie den ansonsten positionsunabhängigen Einbettungen Reihenfolgeinformationen hinzu. Indem den Token-Einbettungen Positionskodierungen hinzugefügt werden, bevor sie in den Transformator eingespeist werden, wird die Position jedes Tokens in der Sequenz für das Modell erkennbar.

Positionskodierungen sind Vektoren, die zu den Einbettungen von Token hinzugefügt werden. Die Intuition besteht darin, diese Vektoren so zu gestalten, dass ihre Werte oder Muster für jede Position eindeutig sind, sodass das Modell zwischen verschiedenen Positionen in der Sequenz unterscheiden kann.

Eine beliebte Methode zur Generierung von Positionskodierungen ist die Verwendung von Sinusfunktionen. Für eine gegebene Position $p$ in der Sequenz und Dimension $d$ der Einbettung wird die Positionskodierung wie folgt berechnet:

$$ PE(2,i) = \sin(\frac{p}{10000^{2i/d}}) $$
$$ PE(2,i+1) = \cos(\frac{p}{10000^{2i/d}}) $$

Wobei $i$ der Dimensionsindex ist. Diese Sinusfunktionen generieren Werte zwischen -1 und 1 und stellen für jede Position ein eindeutiges und wiederholbares Muster sicher.

Die Wahl der Sinusfunktionen ist nicht willkürlich. Sie haben zwei zwingende Eigenschaften:

1. Sie erzeugen Werte zwischen -1 und 1 und sind daher mit den meisten eingebetteten Wertebereichen kompatibel.
2. Ihre Muster ermöglichen es dem Modell, Positionen über die während des Trainings gesehenen Sequenzlängen hinaus zu extrapolieren.

Man könnte sich fragen, warum man die Position des Tokens nicht einfach als Ganzzahl an die Einbettung anfügt oder hinzufügt. Die Herausforderung bei diesem Ansatz ist die Skalierung. Einbettungswerte können, insbesondere nach dem Training, innerhalb eines bestimmten Bereichs existieren, und das direkte Hinzufügen großer Ganzzahlen (für Token weiter unten in langen Sequenzen) kann die Informationen in den ursprünglichen Einbettungen zerstören.

Darüber hinaus wäre die Verwendung von Roh-Ganzzahlen für das Modell keine konsistente Möglichkeit, Sequenzlängen zu verallgemeinern oder zu extrapolieren, die während des Trainings nicht erkannt wurden. Sinusförmige Funktionen bieten dagegen ein vorhersagbares Muster, das bei einer solchen Extrapolation hilft.

Der folgende Code beschreibt eine einfache Implementierung eines transformerbasierten Modells unter Verwendung der integrierten Funktionen von PyTorch. Die Klasse **TransformerModel** kapselt ein transformerbasiertes neuronales Netzwerk, das für die Sequenzverarbeitung entwickelt wurde. Bei der Initialisierung richtet das Modell mehrere Komponenten ein: einen coder zum Anpassen der Eingabedaten an eine gewünschte Dimension, einen **pos_coder**, um der Sequenz Positionsinformationen zu verleihen, einen zentralen **transformer_coder**, der aus mehreren Schichten besteht, um die Sequenz zu verarbeiten, und einen **Decoder** zum Erzeugen der endgültigen Ausgabe. Während die Daten während des Vorwärtsdurchlaufs durch das Modell fließen, durchlaufen sie eine Reihe von Transformationen: Sie werden zuerst in eine höhere Dimension projiziert, dann mit PositionsKodierungen erweitert, von den Transformerschichten verarbeitet und schließlich wird die Darstellung des letzten Tokens genutzt, um die Ausgabe zu erzeugen. Eine Instanz dieses Modells lässt sich leicht erstellen und kann einem Rechengerät zum weiteren Training oder zur Inferenz zugewiesen werden.



In [5]:
# Positionskodierung für Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0), :]
        return self.dropout(x)

# Erstellen des Transformer-Modells

Der folgende Code erstellt das eigentliche transformerbasierte Modell zur Zeitreihenvorhersage. Das Modell ist so aufgebaut, dass es die folgenden Parameter akzeptiert.

* **input_dim**: Die Dimension der Eingabedaten. In diesem Fall verwenden wir nur eine Eingabe, die Anzahl der Sonnenflecken.
* **d_model**: Die Anzahl der Features in den internen Darstellungen des Transformermodells (auch die Größe der Einbettungen). Dies steuert, wie viel ein Modell speichern und verarbeiten kann.
* **nhead**: Die Anzahl der Aufmerksamkeitsköpfe im Multi-Head-Self-Attention-Mechanismus.
* **num_layers**: Die Anzahl der Transformator-coder-Ebenen.
Dropout: Die Dropout-Wahrscheinlichkeit.



In [6]:
# Modelldefinition mit Transformer
class TransformerModel(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super(TransformerModel, self).__init__()

        self.coder = nn.Linear(input_dim, d_model)
        self.pos_coder = PositionalEncoding(d_model, dropout)
        coder_layers = nn.TransformercoderLayer(d_model, nhead)
        self.transformer_coder = nn.Transformercoder(coder_layers, num_layers)
        self.decoder = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.coder(x)
        x = self.pos_coder(x)
        x = self.transformer_coder(x)
        x = self.decoder(x[:, -1, :])
        return x


model = TransformerModel().to(device)

Die Transformer-Architektur in PyTorch wird durch wichtige Konfigurationsentscheidungen bestimmt, von denen **d_model**, **nhead** und **num_layers** eine erhebliche Rolle spielen. Das **d_model** bezeichnet die Dimensionalität der Eingabeeinbettungen und beeinflusst die Fähigkeit des Modells, komplexe Darstellungen zu lernen. Während ein umfangreicheres **d_model** die Reichhaltigkeit des Modellverständnisses stärken kann, erhöht es auch den Rechenaufwand und kann, wenn es nicht sorgfältig ausgewählt wird, ein Überanpassungsrisiko bergen. Parallel dazu werden der Gradientenfluss und die Initialisierung des Modells durch diese Wahl beeinflusst, obwohl die Normalisierungsschichten des Transformers potenzielle Probleme oft abmildern.

Auf der anderen Seite spiegelt **nhead** die Anzahl der Köpfe im Multi-Head-Aufmerksamkeitsmechanismus wider. Eine höhere Anzahl von Köpfen verleiht dem Modell die Fähigkeit, sich gleichzeitig auf verschiedene Segmente der Eingabe zu konzentrieren, wodurch die Erfassung unterschiedlicher kontextueller Nuancen ermöglicht wird. Allerdings gibt es einen Kompromiss. Über einem bestimmten Schwellenwert hinaus kann der Rechenaufwand die marginalen Leistungsgewinne übersteigen. Diese parallele Verarbeitung durch mehrere Aufmerksamkeitsköpfe bietet tendenziell stabilere und vielfältigere Gradienteninformationen, was sich positiv auf die Trainingsdynamik auswirkt.

Schließlich bestimmt der Parameter **num_layers** die Tiefe des Transformers und bestimmt die Anzahl der gestapelten coder- oder Decoder-Ebenen. Ein tieferes Modell kann aufgrund der höheren Anzahl an Ebenen komplexere und hierarchischere Beziehungen in den Daten erkennen. Es gibt jedoch einen Vorbehalt: Ab einer bestimmten Tiefe können potenzielle Leistungsverbesserungen stagnieren und das Risiko einer Überanpassung kann eskalieren. Das Training tieferer Modelle bringt auch eine Reihe von Herausforderungen mit sich. Obwohl Restverbindungen und Normalisierung in Transformern einige Bedenken ausräumen, kann eine hohe Ebenenanzahl Techniken wie Gradienten-Clipping oder Anpassungen der Lernrate für ein stabiles Training erforderlich machen.

Im Wesentlichen stellen diese Parameter ein komplexes Gleichgewicht zwischen Modellkapazität, Rechenleistung und Generalisierungsfähigkeit dar. Ihre optimalen Einstellungen ergeben sich oft aus aufgabenspezifischen Experimenten, der Art der Daten und der verfügbaren Rechenleistung.

## Trainieren des Modells


Das Training eines transformerbasierten Modells folgt vielen der bekannten Paradigmen und Best Practices, die für andere neuronale Netzwerkarchitekturen gelten. Ähnlich wie die Modelle, die wir zuvor kennengelernt haben, profitiert ein transformerbasiertes Modell vom Training in Batches, was sowohl die Rechenleistung als auch die Generalisierung verbessert. Das Training in Batches stellt sicher, dass das Modell seine Gewichte basierend auf dem durchschnittlichen Gradienten über mehrere Datenpunkte aktualisiert, anstatt übermäßig von einer einzelnen Instanz beeinflusst zu werden. Darüber hinaus dient die Verwendung eines frühen Stopps als Schutz vor Überanpassung. Indem wir die Leistung des Modells anhand eines Validierungssatzes überwachen und das Training anhalten, wenn über eine festgelegte Anzahl von Epochen keine signifikante Verbesserung beobachtet wird, stellen wir sicher, dass das Modell gut generalisiert und sich nicht nur die Trainingsdaten merkt. Der Validierungssatz bleibt ein wesentlicher Bestandteil des Trainingsplans, da er ein Proxy-Maß für die Leistung des Modells bei unbekannten Daten bietet und die Feinabstimmung der Hyperparameter steuert. Während Transformerarchitekturen also neue Mechanismen und Komplexitäten einführen, bleiben die grundlegenden Prinzipien des Trainings von Deep-Learning-Modellen in PyTorch konsistent.

In [7]:
# Trainieren des Modells
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, "min", factor=0.5, patience=3, verbose=True)

epochs = 1000
early_stop_count = 0
min_val_loss = float("inf")

for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        x_batch, y_batch = batch
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # Validierung
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in test_loader:
            x_batch, y_batch = batch
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            val_losses.append(loss.item())

    val_loss = np.mean(val_losses)
    scheduler.step(val_loss)

    if val_loss < min_val_loss:
        min_val_loss = val_loss
        early_stop_count = 0
    else:
        early_stop_count += 1

    if early_stop_count >= 5:
        print("Early stopping!")
        break
print(f"Epoche {Epoche + 1}/{Epochen}, Validierungsverlust: {val_loss:.4f}")

Epoch 1/1000, Validation Loss: 0.0421
Epoch 2/1000, Validation Loss: 0.0512
Epoch 3/1000, Validation Loss: 0.0429
Epoch 4/1000, Validation Loss: 0.0376
Epoch 5/1000, Validation Loss: 0.0386
Epoch 6/1000, Validation Loss: 0.0403
Epoch 7/1000, Validation Loss: 0.0358
Epoch 8/1000, Validation Loss: 0.0395
Epoch 9/1000, Validation Loss: 0.0385
Epoch 10/1000, Validation Loss: 0.0368
Epoch 11/1000, Validation Loss: 0.0353
Epoch 12/1000, Validation Loss: 0.0376
Epoch 13/1000, Validation Loss: 0.0390
Epoch 14/1000, Validation Loss: 0.0344
Epoch 15/1000, Validation Loss: 0.0362
Epoch 16/1000, Validation Loss: 0.0408
Epoch 17/1000, Validation Loss: 0.0343
Epoch 18/1000, Validation Loss: 0.0396
Epoch 19/1000, Validation Loss: 0.0357
Epoch 20/1000, Validation Loss: 0.0355
Epoch 00021: reducing learning rate of group 0 to 5.0000e-04.
Epoch 21/1000, Validation Loss: 0.0434
Early stopping!


Wir können jetzt die Leistung dieses Modells bewerten.

In [8]:
# Auswertung
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        x_batch, y_batch = batch
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        predictions.extend(outputs.squeeze().tolist())

rmse = np.sqrt(
    np.mean(
        (
            scaler.inverse_transform(np.array(predictions).reshape(-1, 1))
            - scaler.inverse_transform(y_test.numpy().reshape(-1, 1))
        )
        ** 2
    )
)
print(f"Punktzahl (RMSE): {rmse:.4f}")

Score (RMSE): 15.0481
